# 1. Data Cleaning & Format

In [ ]:
import os

os.environ["HF_TOKEN"] = "SECRET"

In [2]:
from src.preprocess_parquet import load_parquet, preProcess
import pandas as pd

pd.set_option('display.width', 1000) 

if False:
    DROP_TOPIC = [
        'An ninh quốc gia',
        'Cán bộ, công chức, viên chức',
        'Quốc phòng',
        'Chính sách xã hội',
        'Thống kê',
        'Xây dựng pháp luật và thi hành pháp luật',
        'Tổ chức bộ máy nhà nước',
        'Ngoại giao, điều ước quốc tế',
        'Dân số, gia đình, trẻ em, bình đẳng giới',
        'Hình sự',
        'Tổ chức chính trị - xã hội, hội',
        'Tương trợ tư pháp',
        'Dân tộc',
        'Tôn giáo, tín ngưỡng',
        'Văn thư lưu trữ'
    ]

    df = load_parquet("data/dataset.parquet")

    irrelevant_mask = df['topic_title'].isin(DROP_TOPIC)
    df = df[~irrelevant_mask]
    print(f"Drop {sum(irrelevant_mask)} samples theo chủ đề.")
    
    empty_mask = df['content_text'] == ""
    df = df[~empty_mask]
    print(f"Drop {sum(empty_mask)} samples có nội dung trống.")

    df = preProcess(df, save_path='data/processed_dataset.parquet',
                    fix_path = "data/error_correction.csv")

    print(f"Còn lại tổng cộng {len(df)} samples.")
    df

# 2. Retrieving

In [3]:
from src.preprocess_parquet import load
import pandas as pd

pd.set_option('display.width', 1000) 

df = load('data/processed_dataset.parquet')
df = df.drop(columns=['article_title', 'source_note_text', 'source_links', 'topic_title'])
df.head(1)

,docs_code,docs_title,article_index,subject_title,content_text,content_word_count,content_clause_count
947,25/2008/QH12,Luật số 25/2008/QH12,Điều 1,Bảo hiểm y tế,"{'title': '', 'content': [{'text': '1. Luật nà...",135,3


In [4]:
from src.recall_retriever import BM25
from src.data_presentation import Corpus, HierarchicalCorpus

corpus_simple = Corpus()
corpus_hier = HierarchicalCorpus(
    alpha = 0.5,
    title_blending = True
)

bm25_raw = BM25(
    name="BM25_ra",
    corpus= corpus_simple,
    top_k=10,
    norm=0.75
)
bm25_hier = BM25(
    name="BM25_hi",
    corpus= corpus_hier,
    top_k=10,
    norm=0.25
)


In [9]:
from sentence_transformers import SentenceTransformer
from src.recall_retriever import Dense

dense_model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    token=os.environ["HF_TOKEN"]
)

print(f"[INFO] Đã nạp xong dense model lên device={dense_model.device}.")

dense_raw = Dense(
    corpus_simple, dense_model,
    batch_size=32,
    top_k = 100, 
    M=32, ef_construction=512, ef_search=128,
    index_type='HNSW', name='DENSE_ra' 
)

dense_raw_flat = Dense(
    corpus_simple, dense_model,
    batch_size=32,
    top_k = 100, 
    index_type='Flat', name='DENSE_fl' 
)

dense_hier = Dense(
    corpus_hier, dense_model,
    batch_size=256,
    top_k = 2000, 
    M=256, ef_construction=4096, ef_search=2048,
    index_type='HNSW', name='DENSE_ra' 
)

dense_hier_flat = Dense(
    corpus_hier, dense_model,
    batch_size=256,
    top_k = 2000, 
    index_type='Flat', name='DENSE_fl' 
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INFO] Đã nạp xong dense model lên device=cuda:0.


In [10]:
from src.retrieval_pipeline import RetrievalPipeline


retriever = RetrievalPipeline(
    data = df['content_text'],
    retrievalLayers=[dense_raw_flat, dense_hier]
)

[INFO] Đang build Corpuses
[INFO] [1/2] Đang build <class 'src.data_presentation.HierarchicalCorpus'>
[INFO] [2/2] Đang build <class 'src.data_presentation.Corpus'>
[INFO] Đang fit lớp Recall Retrievers
[INFO] [1/2] Đang fit <class 'src.recall_retriever.Dense'> với Knowledge Base <class 'src.data_presentation.Corpus'>


Batches:   0%|          | 0/1303 [00:00<?, ?it/s]

[INFO] [2/2] Đang fit <class 'src.recall_retriever.Dense'> với Knowledge Base <class 'src.data_presentation.HierarchicalCorpus'>


Batches:   0%|          | 0/999 [00:00<?, ?it/s]

# 3. Manual Evaluation

In [15]:
score = retriever.retrieve("Các cơ sở ươm tạo và khu làm việc chung được hưởng những chính sách hỗ trợ nào về thuế và đất đai")

score_board = pd.concat([df[['docs_code', 'article_index']], score], axis=1)

/mnt/082E39432E392AD8/Development Site/Code MESS/VS Code/Python/AI Guru/src/data_presentation.py:216: RuntimeWarning: divide by zero encountered in scalar divide
  mean = np.mean([self.min_penalty / penalty] * penalty + [item[1] for item in score_board])


In [16]:
score_board.sort_values(by="DENSE_ra_score", ascending=False).head(16)

,docs_code,article_index,DENSE_fl_score,DENSE_ra_score,DENSE_ra_comment
8898,04/2017/QH14,Điều 12,0.909680,0.919551,"[3, 2]"
17046,21/2008/QH12,Điều 22,0.908775,0.901958,"[1, 1]"
8901,07/2020/TT-BKHCN,Điều 4,0.870648,0.890830,[1]
17135,10/2024/NĐ-CP,Điều 12,0.895204,0.889607,[2]
17197,74/2017/NĐ-CP,Điều 12,0.887076,0.888162,"[2, 1]"
3271,154/2013/NĐ-CP,Điều 21,0.870353,0.887534,[12]
24219,57/2018/NĐ-CP,Điều 7,0.880391,0.886972,[3]
24179,57/2018/NĐ-CP,Điều 2,0.880877,0.885856,[2]
24112,77/2018/NĐ-CP,Điều 4,0.884264,0.885854,"[1, 1]"
17196,74/2017/NĐ-CP,Điều 11,0.881897,0.885395,[3]


In [17]:
score_board.sort_values(by="DENSE_fl_score", ascending=False).head(16)

,docs_code,article_index,DENSE_fl_score,DENSE_ra_score,DENSE_ra_comment
8898,04/2017/QH14,Điều 12,0.909680,0.919551,"[3, 2]"
17046,21/2008/QH12,Điều 22,0.908775,0.901958,"[1, 1]"
17135,10/2024/NĐ-CP,Điều 12,0.895204,0.889607,[2]
17138,10/2024/NĐ-CP,Điều 15,0.892688,0.778501,[5]
8902,07/2020/TT-BKHCN,Điều 5,0.888460,0.880955,"[1, 2]"
17197,74/2017/NĐ-CP,Điều 12,0.887076,0.888162,"[2, 1]"
17199,74/2017/NĐ-CP,Điều 14,0.886274,0.877408,[1]
24161,40/2017/NĐ-CP,Điều 15,0.885173,0.885172,[]
33344,153/2011/TT-BTC,Điều 1,0.884816,0.731894,[2]
33372,48/2010/QH12,Điều 9,0.884672,0.729480,[7]


In [18]:
df.loc[18195]['content_text']

{'title': 'Người lao động làm thêm giờ vào ban đêm theo khoản 3 Điều 98 của Bộ luật Lao động, được hưởng tiền lương tính theo công thức sau:',
 'content': [{'title': '1. Đối với người lao động hưởng lương theo thời gian, tiền lương làm thêm giờ vào ban đêm được tính như sau: Trong đó:',
   'content': [{'text': 'a) Tiền lương giờ thực trả của công việc đang làm vào ngày làm việc bình thường được xác định theo điểm a khoản 1 Điều 55 Nghị định này;'},
    {'text': 'b) Tiền lương giờ vào ban ngày của ngày làm việc bình thường hoặc của ngày nghỉ hằng tuần hoặc của ngày nghỉ lễ, tết, ngày nghỉ có hưởng lương được xác định như sau: b1) Tiền lương giờ vào ban ngày của ngày làm việc bình thường, được tính ít nhất bằng 100% so với tiền lương giờ thực trả của công việc đang làm vào ngày làm việc bình thường đối với trường hợp người lao động không làm thêm giờ vào ban ngày của ngày đó (trước khi làm thêm giờ vào ban đêm); ít nhất bằng 150% so với tiền lương giờ thực trả của công việc đang làm vào 